In [16]:
import warnings
warnings.filterwarnings('ignore')
# ## MATPLOTLIB SETTINGS
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import cm
font = {'weight' : 'bold','size'   : 22}
mpl.rc('font', **font)
mpl.rc('font',   size=209)  #set defaults so that the plots are readable
mpl.rc('axes',   titlesize=20)
mpl.rc('axes',   labelsize=20)
mpl.rc('xtick',  labelsize=20)
mpl.rc('ytick',  labelsize=20)
mpl.rc('legend', fontsize =20)
mpl.rc('figure', titlesize=20)
mpl.rc('text',   usetex=True)
## FURTHER MATPLOTLIB FEATURES
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.ticker import ScalarFormatter, FormatStrFormatter
from matplotlib.colors import ListedColormap
colors = ['#ff0000', "#41b6c4",  "olivedrab", "darkviolet", '#030764', "#000080"]
my_cmap = ListedColormap(colors, name="my_cmap")
font_dict = {'family': 'serif', 'color':  None,  'weight': 'bold','size': 30}

from cobaya.run import run
from getdist import loadMCSamples
import getdist.plots as gdplt
from getdist import MCSamples
from getdist import plots
#############################################
#BASIC SYSTEM PACKAGES
import sys,time, os, h5py, emcee
from copy import deepcopy as dcopy
from emcee import autocorr
#############################################
import numpy as np
from scipy.stats import chi2 as chi2_func
from scipy.stats import norm as norm_func
from scipy.special import erfcinv
#############################################
#PATH TO SCRIPTS FOLDER
path =  '/data/AMARINS/LSSxHI-CODES/scripts/'
sys.path.insert(1, path)
import cross_functions_theory      as cxft
import cross_functions_theory_4forecast as c4ft
import cross_functions_simulations as cxfs
#############################################
## DEFINITIONS
import pymaster as nmt
EPS = 1e-25
nu_HI   = 1420.405751768 #MHz
C_LIGHT = 299792458 / 1e3  # km/s
STERADIAN = np.radians(1/60) ** 2
#############################################
## NAMASTER SETTINGS 
DEL_L = 4
NSIDE = 256
NPIX  = 12*NSIDE*NSIDE
LMIN,LMAX=30,400
##
b    = nmt.NmtBin.from_nside_linear(NSIDE, nlb=DEL_L)
leff = b.get_effective_ells()
feff = leff*(leff+1)/2/np.pi

In [375]:
def get_flatchain( filename=None, steps_done = -1, burnin=30, verbose=True):   
    ##################  
    backend = emcee.backends.HDFBackend(filename, read_only=True)
    steps_done = steps_done if steps_done>0 else backend.iteration
    with h5py.File(filename, "r") as f:
        chain    = f['mcmc']['chain'][:]     # Shape: (n_steps, n_walkers, n_parameters)
        log_prob = f['mcmc']['log_prob'][:]  # Shape: (n_steps, n_walkers)
    chain    = chain[burnin:steps_done, :,:]
    chain    = chain.reshape(-1, chain.shape[-1]) 
    log_prob = log_prob[burnin:steps_done, :]     
    chi2     = -2 * log_prob.flatten()
    nchains  = chain.shape[1]    
    return {'chain':chain, 'chi2': chi2 }
    
def check_convergence(filename, burnin=0):
    backend = emcee.backends.HDFBackend(filename, read_only=True)
    iteration = backend.iteration
    if iteration == 0:
        return False, iteration, [np.nan]
    samples = backend.get_chain()
    if burnin:
        samples = samples[burnin:,:,:]
        iteration = samples.shape[0]
    flat_samples = samples.reshape(-1, samples.shape[-1])    
    tau_est = []
    for dim in range(flat_samples.shape[1]):
        try:
            tau = autocorr.integrated_time(
                flat_samples[:, dim], 
                c=min(5, iteration//50),  # Adapt window size to chain length
                quiet=True
            )
            tau_est.append(tau)
        except autocorr.AutocorrError:
            tau_est.append(np.nan)
    
    converged = False
    if not np.any(np.isnan(tau_est)):
        max_tau = np.max(tau_est)
        converged = iteration > 100* max_tau
    #print(iteration,100* max_tau)
    if converged and (tau_est[0]<=1 or tau_est[1]<=1):
        converged=False
    return converged, iteration, tau_est

def get_dstat( chi2_data=None,chi2_std=None, dof=2,
               verbose=True):
    p_  = chi2_func.sf(abs(chi2_data-chi2_std), df=dof) 
    confidence_level = 1 - p_
    sigma_2tail = np.sqrt(2) * erfcinv(p_)
    sigma_1tail = norm_func.isf(p_)   
    if verbose:
        print("confidence level   : {:.2e}".format(confidence_level))
        print("sigma (1-tail)     : {:.2e}".format(sigma_1tail))
        print("sigma (2-tail)     : {:.2e}".format(sigma_2tail))    
    return {'delta_pvalue':p_, 'confidence level':confidence_level, 
            'sigma (1-tail)':sigma_1tail, 'sigma (2-tail)':sigma_2tail}
###
def get_stat( filename=None  , run_name='binX', dof=None, 
              steps_done = -1, burnin=30, #binsf1f2=None, 
              chains_info=False, stats_info=False, params_info=False,
              fid_parameter_vec = [-1.0,0.0],
              verbose=True):
    #nperc = 'bin {}'.format(ibin)
    #nch = bins_f1f2[ibins[i]].size
    #dof = 93*nch-2               
    ##################  
    if verbose: print('Data: ',run_name)
    backend = emcee.backends.HDFBackend(filename, read_only=True)
    steps_done = steps_done if steps_done>0 else backend.iteration
    if chains_info and verbose:
        print(50*"-")
        print(f"Current iteration  : {steps_done}")
        print(50*"-")
    with h5py.File(filename, "r") as f:
        chain    = f['mcmc']['chain'][:]     # Shape: (n_steps, n_walkers, n_parameters)
        log_prob = f['mcmc']['log_prob'][:]  # Shape: (n_steps, n_walkers)
    chain    = chain[burnin:steps_done, :,:]
    log_prob = log_prob[burnin:steps_done, :]  
    chi2     = -2 * log_prob.flatten()
    nchains = chain.shape[1]
    if chains_info and verbose:
        print(f"Number of chains   : {nchains}")
        print(f"Burn-in            : {burnin}")
        converged, it, tau = check_convergence(filename, burnin)
        print(f"Iteration          : {it}")
        print(50*"-")
        print("Autocorr times [0] : {:.3f} | current".format(float(tau[0])))
        print("Autocorr times [1] : {:.3f} | baseline".format(float(tau[1])))
        print(f"Converged          : {converged}")
        #print(50*"-")
    chi2_min = chi2.min()
    p_value1  = chi2_func.sf(chi2_min, df=dof) #1-chi2_func.cdf(chi2_min, df=dof) 
    if stats_info and verbose:
        print(50*"-")            
        print("Degree of Freedom[1]: {:.0f}".format(dof))
        print("chi2-min            : {:.0f}".format(chi2_min))
        print("chi2-min-reduced    : {:.2f}".format(chi2_min/dof))
        print("p-value[1]          : {:.2e}".format(p_value1))
        print(50*"-")
        
    sampler = emcee.backends.HDFBackend(filename, read_only=True)
    #################
    chain  = sampler.get_chain(   discard=burnin, thin=10, flat=False)
    lnprob = sampler.get_log_prob(discard=burnin, thin=10, flat=False)
    #################
    chi2              = -2 * lnprob.flatten()
    samples_flat      = chain.reshape(-1, chain.shape[-1])  
    samples_with_chi2 = np.hstack([samples_flat, chi2.reshape(-1, 1)])
    params=['w0','wa','chi2']
    labels=[r'w_0',r'w_a',r'\chi^2']
    sample = MCSamples(
                samples=samples_with_chi2,  # Now includes chi² as a parameter
                names=params,
                labels=labels,
                ignore_rows=0.0,
                settings={'smooth_scale_1D': 0.7, 'smooth_scale_2D': 0.7}
        )      
    chain    = chain[burnin:steps_done, :,:]
    flatchain    = chain.reshape(-1, chain.shape[-1])     
    if stats_info:
        dt = get_significance(samples=flatchain, dof=len(fid_parameter_vec), fid_vec=fid_parameter_vec, verbose=1)
        print(50*"-")  
    else:
        dt = get_significance(samples=flatchain, dof=len(fid_parameter_vec), fid_vec=fid_parameter_vec, verbose=0)    
        
    if params_info and verbose:
        #print(50*"-")
        print(sample.getInlineLatex("w0", limit=1))    
        print(sample.getInlineLatex("wa", limit=1))   
        #print(r"$\chi^2_\min$: " + r"{:.1f}".format(chi2.min()))
        print(50*"-")  
        print()
    return {"pvalue[1]":p_value1, "pvalues[2]":dt['p-value'], 'chi2_min':chi2_min, 'chi2':chi2, 'chain':chain, 'dof':dof}
###
def calculate_kl_gaussian(chain_p, chain_q):
    """
    Calculates KL divergence assuming Multivariate Gaussian distributions.
    chain_p: Samples from Reference (Theoretical)
    chain_q: Samples from Approximation (Simulated)
    """
    mu_p  = np.mean(chain_p, axis=0)
    cov_p = np.cov(chain_p, rowvar=False)
    mu_q = np.mean(chain_q, axis=0)
    cov_q = np.cov(chain_q, rowvar=False)
    #
    k = len(mu_p)#dimensions[3]
    cov_q_inv = np.linalg.inv(cov_q)
    #
    term_trace = np.trace(np.dot(cov_q_inv, cov_p))#trace of (Inv(Sig_Q) * Sig_P) [1]
    #
    diff = mu_q - mu_p
    term_mean = np.dot(diff.T, np.dot(cov_q_inv, diff))#Quadratic difference of means [2]
    #
    det_p = np.linalg.slogdet(cov_p)[1]#Log determinant ratio
    det_q = np.linalg.slogdet(cov_q)[1]#slogdet for numerical stability 
    term_log = det_q - det_p#[4]
    #
    kl_div = 0.5 * (term_trace + term_mean - k + term_log)#[1]+[2]+[3]+[4]
    return kl_div

def calculate_kl_kde(chain_p, chain_q, num_points=100j):
    from scipy.stats import gaussian_kde
    """
    Calculates KL divergence using KDE and numerical integration.
    """
    #grid to integrate
    # Grid needs to cover the extent of BOTH distributions
    x_min = min(chain_p[:,0].min(), chain_q[:,0].min())
    x_max = max(chain_p[:,0].max(), chain_q[:,0].max())
    y_min = min(chain_p[:,1].min(), chain_q[:,1].min())
    y_max = max(chain_p[:,1].max(), chain_q[:,1].max())
    #Small buffer to edges
    X, Y = np.mgrid[x_min:x_max:num_points, y_min:y_max:num_points]
    positions = np.vstack([X.ravel(), Y.ravel()])
    #Estimate PDFs
    # chain_p.T because gaussian_kde expects shape (dims, n_samples)
    kde_p = gaussian_kde(chain_p.T)
    kde_q = gaussian_kde(chain_q.T)
    #PDFs on grid
    #Z_p and Z_q: pdfs
    Z_p = np.reshape(kde_p(positions).T, X.shape)
    Z_q = np.reshape(kde_q(positions).T, X.shape)
    
    #Numerical Integration
    # KL = Sum( P * log(P/Q) * dx * dy )
    epsilon = 1e-15
    mask = (Z_p > epsilon) & (Z_q > epsilon)
    # Calculate integrand only where valid
    integrand = Z_p[mask] * np.log(Z_p[mask] / Z_q[mask])
    # Calculate area element dA
    dx = (x_max - x_min) / (complex(num_points).imag - 1)
    dy = (y_max - y_min) / (complex(num_points).imag - 1)
    dA = dx * dy
    #
    kl_div = np.sum(integrand) * dA
    return kl_div 

def calculate_kl_knn(chain_p, chain_q, k=10):
    from sklearn.neighbors import NearestNeighbors
    """
    Calculates KL divergence D_KL(P || Q) using a kNN (nearest-neighbor) estimator.
    
    chain_p: Samples from Reference P (e.g., Theoretical posterior)  [shape: (N, d)]
    chain_q: Samples from Approximation Q (e.g., Simulated posterior) [shape: (M, d)]
    k: number of neighbors (typical: 5-20 for d=2)
    
    Returns:
        KL divergence in nats.
    
    Notes:
    - KL is NOT symmetric: D_KL(P||Q) != D_KL(Q||P)
    - This implementation uses the Pérez-Cruz style estimator:
        KL ≈ log(M/(N-1)) + d * mean(log(s_i / r_i))
      where r_i is distance to k-th neighbor in P (excluding itself),
      and s_i is distance to k-th neighbor in Q.
    """
    P = np.asarray(chain_p, dtype=float)
    Q = np.asarray(chain_q, dtype=float)

    if P.ndim != 2 or Q.ndim != 2:
        raise ValueError("chain_p and chain_q must be 2D arrays with shape (N, d) and (M, d).")
    if P.shape[1] != Q.shape[1]:
        raise ValueError("chain_p and chain_q must have the same dimensionality (same number of columns).")

    N, d = P.shape
    M, _ = Q.shape

    if N <= k + 1:
        raise ValueError(f"chain_p must have at least k+2 samples. Got N={N}, k={k}.")
    if M <= k:
        raise ValueError(f"chain_q must have at least k+1 samples. Got M={M}, k={k}.")

    # 1) Distances to k-th neighbor in P, excluding the point itself:
    nnP = NearestNeighbors(n_neighbors=k+1, algorithm="auto").fit(P)
    distP, _ = nnP.kneighbors(P)
    r = distP[:, k]  # k-th neighbor, since distP[:,0] is self-distance = 0

    # 2) Distances to k-th neighbor in Q, for points in P:
    nnQ = NearestNeighbors(n_neighbors=k, algorithm="auto").fit(Q)
    distQ, _ = nnQ.kneighbors(P)
    s = distQ[:, k-1]  # k-th neighbor in Q (0-indexed)

    # Numerical safety: avoid log(0) if duplicates exist
    eps = 1e-300
    r = np.maximum(r, eps)
    s = np.maximum(s, eps)

    kl_div = np.log(M / (N - 1.0)) + d * np.mean(np.log(s / r))
    return kl_div

def calculate_kl_knn_both(chain_p, chain_q, k=10):
    """
    Returns D_KL(P||Q), D_KL(Q||P), and a symmetric average 0.5*(KLpq+KLqp).
    """
    kl_pq = calculate_kl_knn(chain_p, chain_q, k=k)
    kl_qp = calculate_kl_knn(chain_q, chain_p, k=k)
    return kl_pq, kl_qp, 0.5 * (kl_pq + kl_qp)

import numpy as np

def calculate_mean_shift_sigma(chain_p, chain_q, cov_mode="pooled", reg=1e-12):
    """
    Mean shift between posteriors in 'sigma units' using Mahalanobis distance.
    
    chain_p: samples from P (e.g. theoretical)
    chain_q: samples from Q (e.g. simulated)
    
    cov_mode:
      - "p": use Cov(P)
      - "q": use Cov(Q)
      - "pooled": use 0.5*(Cov(P)+Cov(Q))  (often a good default)
    
    reg: small diagonal regularization added to covariance for numerical stability.
    
    Returns:
      delta_sigma_2d: sqrt( (mu_p - mu_q)^T Sigma^{-1} (mu_p - mu_q) )
      delta2: the squared distance
      mu_p, mu_q, Sigma_used
    """
    P = np.asarray(chain_p, float)
    Q = np.asarray(chain_q, float)

    mu_p = np.mean(P, axis=0)
    mu_q = np.mean(Q, axis=0)

    cov_p = np.cov(P, rowvar=False)
    cov_q = np.cov(Q, rowvar=False)

    if cov_mode == "p":
        Sigma = cov_p
    elif cov_mode == "q":
        Sigma = cov_q
    elif cov_mode == "pooled":
        Sigma = 0.5 * (cov_p + cov_q)
    else:
        raise ValueError("cov_mode must be one of: 'p', 'q', 'pooled'.")

    # regularize & invert
    d = Sigma.shape[0]
    Sigma_reg = Sigma + reg * np.eye(d)
    Sigma_inv = np.linalg.inv(Sigma_reg)

    diff = mu_p - mu_q
    delta2 = float(diff.T @ Sigma_inv @ diff)
    delta_sigma_2d = np.sqrt(delta2)

    return delta_sigma_2d, delta2, mu_p, mu_q, Sigma
from scipy.stats import chi2, norm

def mean_shift_sigma_1d_equivalent(delta2, dof=2):
    """
    Convert Mahalanobis delta^2 into an equivalent 1D Gaussian sigma.
    """
    p = chi2.sf(delta2, df=dof)      # survival function = P(Chi^2 >= delta2)
    sigma_1d = norm.isf(p / 2.0)     # two-sided equivalent
    return sigma_1d
    
def get_significance(samples=None, dof=2, fid_vec=[-1.0, 0.0], verbose=False):
    from scipy.stats import chi2 as chi2_func
    from scipy.stats import norm as norm_func
    '''
    input:
        samples :  EMCEE chain with shape (N_samples, 2)
        dof     : degree-of-freedom (number of independent fitted parameters)
        fid_vec : fiducal parameter values

    output:
        D: Mahalanobis distance
        p-value: 
        significance: Gaussian Sigma equivalent
    '''
    
    # 1. Setup your data
    # Assume 'samples' is your flat EMCEE chain with shape (N_samples, 2)
    # Example: samples[:,0] is w0, samples[:,1] is wa
    mean_vec = np.mean(samples, axis=0)
    cov_matrix = np.cov(samples, rowvar=False)
    inv_cov = np.linalg.inv(cov_matrix)
    
    # Your theoretical inputs (the Truth)
    truth_vec = np.array(fid_vec)  # Replace with your w0, wa
    
    # 2. Calculate Mahalanobis Distance (D squared)
    delta = mean_vec - truth_vec
    D_squared = delta.T @ inv_cov @ delta
    
    # 3. Convert to Significance (Sigma)
    # We use 2 degrees of freedom (df=2) because we are looking at 2 parameters simultaneously.
    # Step A: Calculate the p-value (probability of seeing a result this extreme or more)
    p_value = chi2_func.sf(D_squared, df=dof)
    
    # Step B: Convert p-value to "Gaussian Sigma equivalent"
    # This answers: "How many sigmas would this p-value be in a standard 1D normal distribution?"
    sigma_significance = norm_func.isf(p_value / 2) # Divide by 2 for two-tailed equivalent


    det_cov = np.linalg.det(cov_matrix)
    
    # 2. Calculate Figure of Merit
    # Note: A larger FoM indicates tighter constraints.
    fom = 1.0 / np.sqrt(det_cov)
    
    
    if verbose:
        #print("Chi-squared distance: {:.2e}".format(D_squared))
        print("Degrees of freedom[2]: {}".format(dof))
        print("D^2                  : {:.3f}".format(D_squared))
        print("p-value[2]           : {:.3f}".format(p_value))
        print("Significance         : {:.2f} σ".format(sigma_significance))
        #print(f"Determinant: {det_cov:.6e}")
        #print(f"Figure of Merit: {fom:.2f}")        
    return {"D": D_squared, 'p-value':p_value, 'significance':sigma_significance, "determinant":det_cov, 'FOM': fom}

In [116]:
bins_f1f2 = {
             1: np.array([8,9        ]),
             2: np.array([12,13,14]),        
             3: np.array([15,16,17,18]),
             4: np.array([19,20,21   ]),
             5: np.array([22,23,24   ]),
             6: np.array([25,26      ]),
             7: np.array([28,29      ])    
             }

In [336]:
ibin=None
print("COMPARISON AMONG BINs")
print()
PATH = {'desktop':'/data/AMARINS/EXTERNAL/DESKTOP/outputs/',
        'laptop' :'/data/AMARINS/EXTERNAL/LAPTOP1/outputs/output/',
        'cosmo1' :'/data/AMARINS/LSSxHI-CODES/notebooks/forecast/output/'
       }
filenames = np.array([
                      PATH['cosmo1']+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(1, 3, '500sims', 'estimated'),
                      PATH['cosmo1']+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(2, 3, '500sims', 'estimated'),
                      PATH['cosmo1']+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(3, 3, '500sims', 'estimated'),
                      PATH['cosmo1']+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(4, 3, '500sims', 'estimated'),
                      PATH['cosmo1']+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(5, 3, '500sims', 'estimated'),    
                      PATH['cosmo1']+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(6, 3, '500sims', 'estimated'),    
                      PATH['cosmo1']+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(7, 3, '500sims', 'estimated'),    
                     ])

ibins = [1,2,3]#,4,5,6,7]
Ntemp = ['bin {}'.format(i) for i in ibins]
##########
params_info = False
stats_info  = True
chains_info = True
##########
chi2_min_array = np.array([])
##########
if 1:
    for i,nperc in enumerate(Ntemp):
        nch = bins_f1f2[ibins[i]].size
        dof = 93*nch-2        
        ##################
        steps_done = -1
        burnin = 30
        ##################
        #print('Filename: ',filename_cl_h5)
        print('Data: ',Ntemp[i])
        filename = filenames[i]
        backend = emcee.backends.HDFBackend(filename, read_only=True)
        steps_done = steps_done if steps_done>0 else backend.iteration
        if chains_info:
            print(50*"-")
            print(f"Current iteration  : {steps_done}")
            print(50*"-")
        ##
        with h5py.File(filename, "r") as f:
            chain    = f['mcmc']['chain'][:]  # Shape: (n_steps, n_walkers, n_parameters)
            log_prob = f['mcmc']['log_prob'][:]  # Shape: (n_steps, n_walkers)
        chain    = chain[burnin:steps_done, :,:]
        log_prob = log_prob[burnin:steps_done, :]  
        chi2     = -2 * log_prob.flatten()
        nchains = chain.shape[1]
        if chains_info:
            print(f"Number of chains   : {nchains}")
            print(f"Burn-in            : {burnin}")
            #print(f"Points to be removed: {(burnin)*nchains}")
            #print(f"Points used         : {(steps_done-burnin)*nchains}")
            converged, it, tau = check_convergence(filename, burnin)
            print(f"Iteration          : {it}")
            print(50*"-")
            print("Autocorr times [0] : {:.3f} | current".format(float(tau[0])))
            print("Autocorr times [1] : {:.3f} | baseline".format(float(tau[1])))
            print(f"Converged          : {converged}")
            #print(50*"-")
        
        #print()
        #print(50*"-")
        chi2_min = chi2.min()
        chi2_min_array = np.hstack((chi2_min_array,chi2_min))
        p_value  = chi2_func.sf(chi2_min, df=dof) 
        #p_value  = 1-chi2_func.cdf(chi2_min, df=dof) 
        if stats_info:
            print(50*"-")            
            print("Degree of Freedom  : {:.0f}".format(dof))
            print("chi2-min           : {:.0f}".format(chi2_min))
            print("chi2-min-reduced   : {:.2f}".format(chi2_min/dof))
            print("p-value            : {:.2e}".format(p_value))
            print(50*"-")
                
        filename = filenames[i]
        sampler = emcee.backends.HDFBackend(filename, read_only=True)
        ####
        chain  = sampler.get_chain(   discard=burnin, thin=10, flat=False)
        lnprob = sampler.get_log_prob(discard=burnin, thin=10, flat=False)
        #################
        chi2              = -2 * lnprob.flatten()
        samples_flat      = chain.reshape(-1, chain.shape[-1])  
        samples_with_chi2 = np.hstack([samples_flat, chi2.reshape(-1, 1)])
        params=['w0','wa','chi2']
        labels=[r'w_0',r'w_a',r'\chi^2']
        sample = MCSamples(
                samples=samples_with_chi2,  # Now includes chi² as a parameter
                names=params,
                labels=labels,
                ignore_rows=0.0,
                settings={'smooth_scale_1D': 0.7, 'smooth_scale_2D': 0.7}
        )      
        if params_info:
            print(50*"-")
            print(sample.getInlineLatex("w0", limit=1))    
            print(sample.getInlineLatex("wa", limit=1))   
            #print(r"$\chi^2_\min$: " + r"{:.1f}".format(chi2.min()))
            print(50*"-")  
            print()


COMPARISON AMONG BINs

Data:  bin 1
--------------------------------------------------
Current iteration  : 5020
--------------------------------------------------
Number of chains   : 10
Burn-in            : 30
Iteration          : 4990
--------------------------------------------------
Autocorr times [0] : 0.963 | current
Autocorr times [1] : 0.952 | baseline
Converged          : False
--------------------------------------------------
Degree of Freedom  : 184
chi2-min           : 224
chi2-min-reduced   : 1.22
p-value            : 2.41e-02
--------------------------------------------------
Data:  bin 2
--------------------------------------------------
Current iteration  : 3950
--------------------------------------------------
Number of chains   : 10
Burn-in            : 30
Iteration          : 3920
--------------------------------------------------
Autocorr times [0] : 0.962 | current
Autocorr times [1] : 0.961 | baseline
Converged          : False
---------------------------------

In [337]:
PATH = {'desktop':'/data/AMARINS/EXTERNAL/DESKTOP/outputs/',
        'laptop' :'/data/AMARINS/EXTERNAL/LAPTOP1/outputs/output/',
        'cosmo1' :'/data/AMARINS/LSSxHI-CODES/notebooks/forecast/output/'
       }
filenames = {
             1:{"theory"   : PATH['desktop']+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(1, 3, '500sims', 'theory'),
                #"simulated": PATH['cosmo1']+"emcee_cx_cpl_{}bin__oldcov__hi_error_100__2000sims_simulated.h5".format(1),
                "simulated": PATH['cosmo1' ]+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(1, 3, '500sims', 'simulated'),
                "estimated": PATH['cosmo1' ]+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(1, 3, '500sims', 'estimated'),
                },
             2:{"theory"   : PATH['desktop']+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(2, 3, '500sims', 'theory'),
                #"simulated": PATH['desktop']+"emcee_cx_cpl_{}bin__oldcov__hi_error_100__2000sims_simulated.h5".format(2),
                "simulated": PATH['cosmo1' ]+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(2, 3, '500sims', 'simulated'),                
                "estimated": PATH['cosmo1' ]+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(2, 3, '500sims', 'estimated'),
                },    
             3:{"theory"   : PATH['desktop']+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(3, 3, '500sims', 'theory'),
                #"simulated": PATH['desktop']+"emcee_cx_cpl_{}bin__oldcov__hi_error_100__2000sims_simulated.h5".format(3),
                "simulated": PATH['cosmo1' ]+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(3, 3, '500sims', 'simulated'),                
                "estimated": PATH['cosmo1' ]+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(3, 3, '500sims', 'estimated'),
                },
             4:{"theory"   : PATH['desktop']+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(4, 3, '500sims', 'theory'),
                #"simulated": PATH['desktop']+"emcee_cx_cpl_{}bin__oldcov__hi_error_100__2000sims_simulated.h5".format(4),
                "simulated": PATH['cosmo1' ]+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(4, 3, '500sims', 'simulated'),                
                "estimated": PATH['cosmo1' ]+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(4, 3, '500sims', 'estimated'),
                },    
             5:{"theory"   : PATH['desktop']+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(5, 3, '500sims', 'theory'),
                #"simulated": PATH['desktop']+"emcee_cx_cpl_{}bin__oldcov__hi_error_100__2000sims_simulated.h5".format(5),
                "simulated": PATH['cosmo1' ]+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(5, 3, '500sims', 'simulated'),                
                "estimated": PATH['cosmo1' ]+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(5, 3, '500sims', 'estimated'),
                },    
             6:{"theory"   : PATH['desktop']+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(6, 3, '500sims', 'theory'),
                #"simulated": PATH['desktop']+"emcee_cx_cpl_{}bin__oldcov__hi_error_100__2000sims_simulated.h5".format(6),
                "simulated": PATH['cosmo1' ]+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(6, 3, '500sims', 'simulated'),                
                "estimated": PATH['cosmo1' ]+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(6, 3, '500sims', 'estimated'),
                },        
             7:{"theory"   : PATH['desktop']+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(7, 3, '500sims', 'theory'),
                #"simulated": PATH['desktop']+"emcee_cx_cpl_{}bin__oldcov__hi_error_100__2000sims_simulated.h5".format(7),
                "simulated": PATH['cosmo1' ]+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(7, 3, '500sims', 'simulated'),
                "estimated": PATH['cosmo1' ]+"emcee_cx_cpl_{}bin__COVmodel{}__hi_error_100__{}_{}.h5".format(7, 3, '500sims', 'estimated'),
                },            
            }
filenames.keys()

dict_keys([1, 2, 3, 4, 5, 6, 7])

In [356]:
verbose,cinfo,sinfo,pinfo=1,0,1,1
plot_thr,plot_sim,plot_est=0,0,0
burnin=30; steps_done=-1
for ibin in [1, 2, 3, 4, 5, 6, 7]:
    print(50*"=")   
    print('BIN {}'.format(ibin))
    #print(50*"=")   
    nch = bins_f1f2[ibin].size
    dof = 93*nch-2    
    try:
        if plot_thr:
            gthr = get_stat( filename=filenames[ibin]['theory']  , run_name='theory'.format(ibin), dof=dof, 
                             steps_done = steps_done, burnin=burnin,
                             chains_info=cinfo, stats_info=sinfo, params_info=pinfo,
                             verbose=verbose )
            print()
    except:
        print(50*'*')        
        print('[skipped] thr bin ',ibin)
        print(50*'*')        
        print()
    try:
        if plot_sim:
            gsim = get_stat( filename=filenames[ibin]['simulated']  , run_name=' simulated'.format(ibin), dof=dof, 
                             steps_done = steps_done, burnin=burnin,
                             chains_info=cinfo, stats_info=sinfo, params_info=pinfo,
                             verbose=verbose )
            print()
    except:
        print(50*'*')        
        print('[skipped] sim bin ',ibin)            
        print(50*'*')    
        print()        
    try:
        if plot_est:
            gest = get_stat( filename=filenames[ibin]['estimated']  , run_name='estimated'.format(ibin), dof=dof, 
                             steps_done = steps_done, burnin=burnin,
                             chains_info=cinfo, stats_info=sinfo, params_info=pinfo,
                             verbose=verbose )
            print()
    except:
        print(50*'*')
        print('[skipped] est bin ',ibin)            
        print(50*'*')
        print()        
    print('Xi2est - Xi2sim:', np.around( (gest['chi2_min']-gsim['chi2_min'])/dof, decimals=3))
    print('sqrt(2*dof)', np.around(np.sqrt(2*dof),decimals=3))
    print()
    print()

BIN 1
Xi2est - Xi2sim: 0.467
sqrt(2*dof) 19.183


BIN 2
Xi2est - Xi2sim: 0.31
sqrt(2*dof) 23.537


BIN 3
Xi2est - Xi2sim: 0.232
sqrt(2*dof) 27.203


BIN 4
Xi2est - Xi2sim: 0.31
sqrt(2*dof) 23.537


BIN 5
Xi2est - Xi2sim: 0.31
sqrt(2*dof) 23.537


BIN 6
Xi2est - Xi2sim: 0.467
sqrt(2*dof) 19.183


BIN 7
Xi2est - Xi2sim: 0.467
sqrt(2*dof) 19.183




In [352]:
gest.keys(), gest['dof']

(dict_keys(['pvalue[1]', 'pvalues[2]', 'chi2_min', 'chi2', 'chain', 'dof']),
 184)

In [344]:
1-np.sqrt(2/277)

0.9150281422675826

In [248]:
thr_info = True
sim_info = True
for ibin in [1, 2, 3, 4, 5, 6, 7]:
    print(50*"=")   
    print('BIN {}'.format(ibin))
    #print(50*"=")   
    nch = bins_f1f2[ibin].size
    dof = 93*nch-2         
    gthr = get_stat( filename=filenames[ibin]['theory']  , run_name='theory'.format(ibin), dof=dof, 
                     steps_done = -1, burnin=30, #binsf1f2=None, 
                     chains_info=0, stats_info=1, params_info=1,
                     verbose=0 )
    gsim = get_stat( filename=filenames[ibin]['simulated']  , run_name='simulated'.format(ibin), dof=dof, 
                     steps_done = -1, burnin=30, #binsf1f2=None, 
                     chains_info=0, stats_info=1, params_info=1,
                     verbose=0 )
    gest = get_stat( filename=filenames[ibin]['estimated']  , run_name='estimated'.format(ibin), dof=dof, 
                     steps_done = -1, burnin=30, #binsf1f2=None, 
                     chains_info=0, stats_info=1, params_info=1,
                     verbose=0 )

    samples_thr_params = gthr['chain'].reshape((-1, 2))
    samples_sim_params = gsim['chain'].reshape((-1, 2))
    samples_est_params = gest['chain'].reshape((-1, 2))

    if thr_info:
        # KL -- Gaussian
        kl_result = calculate_kl_gaussian(samples_thr_params, samples_est_params)
        sigma_tension = np.sqrt(2*kl_result)
        print(f"Gau KL Divergence: {kl_result:.4f} nats | {sigma_tension:.2f}σ")
        # KL -- KDE
        #kl_kde_result = calculate_kl_kde(samples_thr_params, samples_est_params)
        #sigma_tension = np.sqrt(2*kl_kde_result)
        #print(f"KDE KL Divergence: {kl_kde_result:.4f} nats | {sigma_tension:.2f}σ") 
        # KL --KNN
        #kl_knn_result = calculate_kl_knn(samples_thr_params, samples_est_params, k=10)
        #sigma_tension = np.sqrt(2*kl_knn_result)
        #print(f"KNN KL Divergence: {kl_knn_result:.4f} nats | {sigma_tension:.2f}σ")           
        #sigma_tension, _, _, _, _ = calculate_mean_shift_sigma(samples_thr_params, samples_est_params, cov_mode="pooled")
        #print(f"sigma units' using Mahalanobis distance: {sigma_tension:.2f}σ")          
        #print()
    if sim_info:
        # KL -- Gaussian        
        kl_result = calculate_kl_gaussian(samples_sim_params, samples_est_params)
        sigma_tension = np.sqrt(2*kl_result)
        print(f"Gau KL Divergence: {kl_result:.4f} nats | {sigma_tension:.2f}σ")
        # KL -- KDE
        #kl_kde_result = calculate_kl_kde(samples_sim_params, samples_est_params)
        #sigma_tension = np.sqrt(2*kl_kde_result)
        #print(f"KDE KL Divergence: {kl_kde_result:.4f} nats | {sigma_tension:.2f}σ") 
        # KL --KNN
        #kl_knn_result = calculate_kl_knn(samples_sim_params, samples_est_params, k=10)
        #sigma_tension = np.sqrt(2*kl_knn_result)
        #print(f"KNN KL Divergence: {kl_knn_result:.4f} nats | {sigma_tension:.2f}σ")  
        #sigma_tension, _, _, _, _ = calculate_mean_shift_sigma(samples_sim_params, samples_est_params, cov_mode="pooled")
        #print(f"sigma units' using Mahalanobis distance: {sigma_tension:.2f}σ")  
        #print()

BIN 1
Gau KL Divergence: 1.0700 nats | 1.46σ
Gau KL Divergence: 0.9144 nats | 1.35σ
BIN 2
Gau KL Divergence: 0.4147 nats | 0.91σ
Gau KL Divergence: 1.2079 nats | 1.55σ
BIN 3
Gau KL Divergence: 0.3319 nats | 0.81σ
Gau KL Divergence: 0.1637 nats | 0.57σ
BIN 4
Gau KL Divergence: 1.1441 nats | 1.51σ
Gau KL Divergence: 0.0244 nats | 0.22σ
BIN 5
Gau KL Divergence: 0.5582 nats | 1.06σ
Gau KL Divergence: 0.2182 nats | 0.66σ
BIN 6
Gau KL Divergence: 2.8417 nats | 2.38σ
Gau KL Divergence: 0.1607 nats | 0.57σ
BIN 7
Gau KL Divergence: 0.0931 nats | 0.43σ
Gau KL Divergence: 0.2947 nats | 0.77σ


## ALL BINS

In [357]:
flatchain  = get_flatchain( filename=filenames[1]['theory'] , steps_done = -1, burnin=300)['chain']
flatchain.shape

(67000, 2)

In [362]:
for i in range(1,8,1):
    flatchain  = get_flatchain( filename=filenames[i]['estimated'] , steps_done = -1, burnin=300)['chain']
    get_significance(samples=flatchain, dof=2, fid_vec=[-1.0, 0.0], verbose=1);
    print()

Degrees of freedom[2]: 2
p-value[2]           : 0.185
Significance         : 1.33 σ

Degrees of freedom[2]: 2
p-value[2]           : 0.501
Significance         : 0.67 σ

Degrees of freedom[2]: 2
p-value[2]           : 0.497
Significance         : 0.68 σ

Degrees of freedom[2]: 2
p-value[2]           : 0.285
Significance         : 1.07 σ

Degrees of freedom[2]: 2
p-value[2]           : 0.635
Significance         : 0.47 σ

Degrees of freedom[2]: 2
p-value[2]           : 0.118
Significance         : 1.56 σ

Degrees of freedom[2]: 2
p-value[2]           : 0.904
Significance         : 0.12 σ



In [363]:
##########
nch = 19
dof = 93*nch-2
PATH = {'desktop':'/data/AMARINS/EXTERNAL/DESKTOP/outputs/',
        'laptop' :'/data/AMARINS/EXTERNAL/LAPTOP1/outputs/output/',
        'cosmo1' :'/data/AMARINS/LSSxHI-CODES/notebooks/forecast/output/'
       }
filenames_all = {}
filenames_all['theory'   ]=PATH['laptop']+"emcee_cx_cpl_ALLbins__COVmodel{}__hi_error_100__500sims_{}.h5".format(3, 'theory'          )
filenames_all['simulated']=PATH['laptop']+"emcee_cx_cpl_ALLbins__COVmodel{}__hi_error_100__500sims_{}.h5".format(3, 'simulated'       )
filenames_all['estimated']=PATH['laptop']+"emcee_cx_cpl_ALLbins__COVmodel{}__hi_error_100__500sims_{}.h5".format(3, 'estimated__huber')

In [376]:
flatchain  = get_flatchain( filename=filenames_all['estimated'] , steps_done = -1, burnin=300)['chain']
print('Comparison 1')
dt = get_significance(samples=flatchain, dof=2, fid_vec=[-1.0, 0.0], verbose=1);
print()
print('Comparison 2')
dt = get_significance(samples=flatchain, dof=2, fid_vec=[-0.995, -0.02], verbose=1);
print()
print('Comparison 3')
dt = get_significance(samples=flatchain, dof=2, fid_vec=[-1.012, 0.04], verbose=1);

Comparison 1
Degrees of freedom[2]: 2
D^2                  : 0.894
p-value[2]           : 0.639
Significance         : 0.47 σ

Comparison 2
Degrees of freedom[2]: 2
D^2                  : 0.362
p-value[2]           : 0.835
Significance         : 0.21 σ

Comparison 3
Degrees of freedom[2]: 2
D^2                  : 2.250
p-value[2]           : 0.325
Significance         : 0.99 σ


In [365]:
dt.keys()

dict_keys(['D', 'p-value', 'significance', 'determinant', 'FOM'])

In [367]:
flatchain.shape

(39530, 2)

In [342]:
verbose,cinfo,sinfo,pinfo=1,0,1,1
plot_thr,plot_sim,plot_est=1,1,1
burnin=300; steps_done=-1
####
try:
    if plot_thr:
        iname='theory'
        gthr = get_stat( filename=filenames_all[iname]  , run_name=iname, dof=93*nch-2, 
                         steps_done = steps_done, burnin=burnin,
                         chains_info=cinfo, stats_info=sinfo, params_info=pinfo,
                         verbose=verbose )
        chi2_min_thr = gthr['chi2_min']
        #flatchain  = get_flatchain( filename=filenames[i][iname] , steps_done = steps_done, burnin=burnin)['chain']
        #dt = get_significance(samples=flatchain, dof=2, fid_vec=[-1.0, 0.0], verbose=0);        
        #print(f"P-value: {dt['p-value']:.5f}")
        #print(f"Significance: {dt['significance']:.2f} σ")             
        print()
except:
    print(50*'*')        
    print('[skipped] thr')
    print(50*'*')

####
try:
    if plot_sim:
        iname='simulated'
        gsim = get_stat( filename=filenames_all[iname]  , run_name=iname, dof=93*nch-2, 
                         steps_done = steps_done, burnin=burnin,
                         chains_info=cinfo, stats_info=sinfo, params_info=pinfo,
                         verbose=verbose )
        chi2_min_sim = gsim['chi2_min']           
        print()
except:
    print(50*'*')        
    print('[skipped] sim')
    print(50*'*') 

####
try:
    if plot_est:
        iname='estimated'
        gest = get_stat( filename=filenames_all[iname]  , run_name=iname, dof=93*nch-2, 
                         steps_done = steps_done, burnin=burnin,
                         chains_info=cinfo, stats_info=sinfo, params_info=pinfo,
                         verbose=verbose )
        chi2_min_est = gest['chi2_min']     
             
        print()
except:
    print(50*'*')        
    print('[skipped] est')
    print(50*'*')     

Data:  theory
--------------------------------------------------
Degree of Freedom[1]: 1765
chi2-min            : 0
chi2-min-reduced    : 0.00
p-value[1]          : 1.00e+00
--------------------------------------------------
Degrees of freedom[2]: 2
p-value[2]           : 9.19e-01
Significance         : 0.10 σ
--------------------------------------------------
w_0 = -0.995\pm 0.064
w_a = -0.02\pm 0.17
--------------------------------------------------


Data:  simulated
--------------------------------------------------
Degree of Freedom[1]: 1765
chi2-min            : 803
chi2-min-reduced    : 0.45
p-value[1]          : 1.00e+00
--------------------------------------------------
Degrees of freedom[2]: 2
p-value[2]           : 9.65e-01
Significance         : 0.04 σ
--------------------------------------------------
w_0 = -1.012\pm 0.066
w_a = 0.04\pm 0.17
--------------------------------------------------


Data:  estimated
--------------------------------------------------
Degree of Fr

In [272]:
chi2_min = dcopy( chi2_min_thr )
for i,(chi2_min_std, char) in enumerate(zip([chi2_min_sim, chi2_min_est], ['thr','sim'])):
    p_std  = chi2_func.sf(abs(chi2_min-chi2_min_std), df=2) 
    confidence_level = 1 - p_std
    sigma_2tail = np.sqrt(2) * erfcinv(p_std)
    sigma_1tail = norm_func.isf(p_std)   
    print(char.upper())
    print("confidence level   : {:.2e}".format(confidence_level))
    print("sigma (1-tail)     : {:.2e}".format(sigma_1tail))
    print("sigma (2-tail)     : {:.2e}".format(sigma_2tail))
    print()

THR
confidence level   : 1.00e+00
sigma (1-tail)     : 2.82e+01
sigma (2-tail)     : 2.82e+01

SIM
confidence level   : 1.00e+00
sigma (1-tail)     : 3.71e+01
sigma (2-tail)     : 3.71e+01



In [276]:
gthr['chain'].shape

(332, 10, 2)

In [275]:
samples_thr_params = gthr['chain'].reshape((-1, 2))
samples_sim_params = gsim['chain'].reshape((-1, 2))
samples_est_params = gest['chain'].reshape((-1, 2))
#
kl_result = calculate_kl_gaussian(samples_thr_params, samples_est_params)
sigma_tension = np.sqrt(2*kl_result)
print(f"Gau KL Divergence: {kl_result:.4f} nats | {sigma_tension:.2f}σ")

#
kl_result = calculate_kl_gaussian(samples_sim_params, samples_est_params)
sigma_tension = np.sqrt(2*kl_result)
print(f"Gau KL Divergence: {kl_result:.4f} nats | {sigma_tension:.2f}σ")

Gau KL Divergence: 0.3097 nats | 0.79σ
Gau KL Divergence: 0.6781 nats | 1.16σ


In [ ]:
mean_tru = np.array([-1.000, 0.00])
mean_thr = np.array([-0.995, -0.02])
mean_sim = np.array([-1.012, 0.04])
mean_est = np.array([-1.026, 0.05])



In [372]:
# 1. Load your means (Best fit values)
# Example: [w0, wa]
mean_gauss = np.array([-1.05, 0.10])  # From Gaussian Sim
mean_obs   = np.array([-1.11, 0.22])  # From Estimated/Processed Sim

# 2. Load the Covariance Matrix (from the Estimated/Observed chain)
# We use the 'Observed' covariance because we want to know if the bias 
# is significant relative to the final error bars.
cov_obs = np.array([[0.03, -0.05], 
                    [-0.05, 0.15]]) # Example covariance
inv_cov = np.linalg.inv(cov_obs)

# 3. Calculate the Shift Vector (The Systematic Bias)
delta = mean_obs - mean_gauss  # This removes the shared cosmic variance

# 4. Calculate Mahalanobis Distance (D squared)
D_squared = delta.T @ inv_cov @ delta

# 5. Convert to Sigma Significance (for 2 parameters)
# We ask: "How rare would a random statistical jump of this size be?"
p_value = chi2_func.sf(D_squared, df=2)
sigma_bias = norm_func.isf(p_value / 2)

print(f"Parameter Shift (w0, wa): {delta}")
print(f"Bias Significance: {sigma_bias:.2f} σ")

Parameter Shift (w0, wa): [-0.06  0.12]
Bias Significance: 0.08 σ
